# 01 — Explore Data

One-off exploration of PhysioNet EEGBCI: sanity-check loading, preprocessing choices, and event/label structure before writing the real ingestion pipeline in `src/ingest.py`.

This notebook is exploratory only — logic that ends up being used for real (filtering parameters, event-mapping fix, epoch window) lives in `src/`, not here.

## Imports

In [ ]:
import mne
from mne.io import read_raw_edf, concatenate_raws
import random
import numpy as np
import matplotlib.pyplot as plt


## Subject split (seeded)

Quick check that the seeded shuffle is deterministic and excludes the known problem subjects (88, 89, 92, 100 — sampling-rate/annotation issues).

In [ ]:
subjects = [i for i in range(1, 110) if i not in [88, 89, 92, 100]]
random.seed(1234)
random.shuffle(subjects)
print(f"{len(subjects)} usable subjects")
print("first 10:", subjects[:10])


## Load one subject, inspect signal + spectrum

Loading subjects 1–2, runs 4/8/12 (imagined left/right fist) as a small sample to check preprocessing before running it across all 105 subjects.

**Sanity check added:** assert sampling rate is 160 Hz — this is exactly the kind of check that would have caught the 88/89/92/100 issue immediately instead of a silent shape mismatch downstream.

In [ ]:
sample_subjects = [1, 2]
sample_runs = [4, 8, 12]

file_paths = mne.datasets.eegbci.load_data(sample_subjects, sample_runs)
raw_objects = [read_raw_edf(fp, preload=True) for fp in file_paths]
raw = concatenate_raws(raw_objects)

assert raw.info['sfreq'] == 160, f"unexpected sfreq: {raw.info['sfreq']}"
print(raw.info)


### Before filtering — raw power spectrum

Check for the expected 60 Hz US powerline artifact (PhysioNet EEGBCI was recorded in the US — 60 Hz, *not* 50 Hz, which is the European/Asian convention. Easy to get this backwards if copying a notch-filter snippet from a non-US EEG tutorial).

In [ ]:
raw.compute_psd().plot()
plt.suptitle("Before filtering")
plt.show()


### Apply bandpass + notch filter

- Bandpass 8–30 Hz: standard motor-imagery band (covers mu ~8-12Hz and beta ~13-30Hz rhythms), matches EEGNet baseline preprocessing.
- Notch filter at **60 Hz**

In [ ]:
raw.filter(l_freq=8, h_freq=30)
raw.notch_filter(freqs=60)

raw.compute_psd().plot()
plt.suptitle("After bandpass (8-30 Hz) + 60 Hz notch")
plt.show()


## Event extraction — check actual event codes

**Important:** don't hardcode `{'rest': 1, 'left_fist': 2, 'right_fist': 3}`. `concatenate_raws` can insert boundary annotations (`BAD boundary`, `EDGE boundary`) that sort alphabetically before `T0/T1/T2`, shifting the actual integer codes MNE assigns. Always inspect `event_id_dict` directly and build the mapping from it.

In [ ]:
events, event_id_dict = mne.events_from_annotations(raw)
print("actual event_id_dict:", event_id_dict)
print("first 10 events:\n", events[:10])


## Epoch the data

Mapping built from `event_id_dict`'s real values, not assumed integers. `tmax` set to check against actual annotation duration below before trusting a round number like 4.0 or 5.0 seconds.

In [ ]:
# check real trial duration before picking tmax
print("annotation durations (sec):", np.unique(raw.annotations.duration))


In [ ]:
mapping = {
    'rest': event_id_dict['T0'],
    'left_fist': event_id_dict['T1'],
    'right_fist': event_id_dict['T2'],
}
print("mapping (built from actual codes):", mapping)

TMIN, TMAX = 0.0, 4.0  # update if annotation duration check above says otherwise
epochs = mne.Epochs(raw, events, event_id=mapping, tmin=TMIN, tmax=TMAX,
                    baseline=None, preload=True)
print(epochs)


## Confirm output shapes/types before wiring into `src/`

In [ ]:
X = epochs.get_data().astype(np.float32)  # cast explicitly -- avoid float64 bloat
y = epochs.events[:, -1]

print("X shape:", X.shape, X.dtype)
print("y shape:", y.shape, y.dtype)
print("unique labels:", np.unique(y))


## Conclusions -> what goes into `src/`

- Bandpass 8-30 Hz + **60 Hz** notch confirmed against the PSD plots above.
- Event mapping must be built from `event_id_dict`, never hardcoded — confirmed the boundary-annotation risk is real for concatenated raws.
- `tmin=0.0, tmax=4.0` pending confirmation against `raw.annotations.duration` (checked above) — matches ~4.1s PhysioNet trial length without bleeding into the next trial.
- Cast to `float32` explicitly at save time to avoid `float64` size inflation.
- Next: move this into `src/tokenizer.py` / `src/ingest.py` as reusable functions, looped over the full 105-subject split, not just subjects 1-2.